# Combined Kaggle Mushrooms Dataset
This notebook demonstrates how to explore and load the dataset for image classification tasks.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

_KAGGLE_PATH = Path('/kaggle/input/combined-kaggle-mushrooms-dataset/images')
_LOCAL_PATH = Path('images')

if not _KAGGLE_PATH.exists() and not _LOCAL_PATH.exists():
    import kagglehub
    path = kagglehub.dataset_download('dariobaumberger/combined-kaggle-mushrooms-dataset')
    _LOCAL_PATH = Path(path) / 'images'

DATASET_PATH = _KAGGLE_PATH if _KAGGLE_PATH.exists() else _LOCAL_PATH
print(f'Using dataset path: {DATASET_PATH.resolve()}')

## Explore the Dataset

In [ ]:
species = sorted([d.name for d in DATASET_PATH.iterdir() if d.is_dir()])
image_counts = {s: len(list((DATASET_PATH / s).glob('*.webp'))) for s in species}

print(f'Species:      {len(species)}')
print(f'Total images: {sum(image_counts.values())}')
print(f'Avg per species: {sum(image_counts.values()) / len(species):.0f}')

In [ ]:
# Top 20 species by image count
top = sorted(image_counts.items(), key=lambda x: x[1], reverse=True)[:20]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh([s for s, _ in top], [c for _, c in top])
ax.set_xlabel('Image Count')
ax.set_title('Top 20 Species by Image Count')
plt.tight_layout()
plt.show()

## Sample Images

In [ ]:
sample_species = np.random.choice(species, 12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, s in zip(axes.flat, sample_species):
    img_path = next((DATASET_PATH / s).glob('*.webp'))
    ax.imshow(Image.open(img_path))
    ax.set_title(s, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Load with PyTorch

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = ImageFolder(DATASET_PATH, transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

print(f'Classes: {len(dataset.classes)}')
print(f'Images:  {len(dataset)}')
print(f'Batches: {len(loader)}')

## Load with Keras

In [ ]:
import numpy as np
import tensorflow as tf
from PIL import Image

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

class_names = sorted([d.name for d in DATASET_PATH.iterdir() if d.is_dir()])
label_index = {name: i for i, name in enumerate(class_names)}

paths, labels = [], []
for cls in class_names:
    for p in (DATASET_PATH / cls).glob("*.webp"):
        paths.append(str(p))
        labels.append(label_index[cls])


def load(path, label):
    img = tf.numpy_function(
        lambda p: np.array(Image.open(p.decode()).convert("RGB").resize(IMAGE_SIZE)),
        [path],
        tf.uint8,
    )
    img.set_shape([*IMAGE_SIZE, 3])
    return tf.cast(img, tf.float32) / 255.0, tf.one_hot(label, len(class_names))


dataset_keras = (
    tf.data.Dataset.from_tensor_slices((paths, labels))
    .map(load, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
dataset_keras.class_names = class_names

print(f'Classes: {dataset_keras.class_names[:5]} ...')
print(f'Batches: {len(dataset_keras)}')